In [1]:
# ============================================================
# HÜCRE 1: Veri Yükleme ve Ham İnceleme
# ============================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Dosyayı yükle
df_raw = pd.read_excel('mergeAllData.xlsx')

print("=" * 55)
print("  HAM VERİ RAPORU")
print("=" * 55)
print(f"  Toplam Satır        : {len(df_raw):,}")
print(f"  Toplam Sütun        : {df_raw.shape[1]}")
print(f"  Benzersiz Hasta No  : {df_raw['Hasta No'].nunique():,}")
print(f"  HbA1c Dolu Satır    : {df_raw['Hemoglobin (Hb A1c)'].notna().sum():,}")
print("=" * 55)

print("\n📋 Sütunlar ve Eksik Veri Oranları:")
print("-" * 45)
for col in df_raw.columns:
    missing = df_raw[col].isna().sum()
    pct = missing / len(df_raw) * 100
    bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col:<40} %{pct:5.1f}  {bar}")

print("\n📌 İlk 3 Satır (Ham Hali):")
print(df_raw.head(3).to_string())

print("\n✅ Hücre 1 tamamlandı. Çıktıyı paylaşın → Hücre 2'ye geçelim.")

  HAM VERİ RAPORU
  Toplam Satır        : 43,460
  Toplam Sütun        : 12
  Benzersiz Hasta No  : 22,375
  HbA1c Dolu Satır    : 25,195

📋 Sütunlar ve Eksik Veri Oranları:
---------------------------------------------
  Hasta No                                 %  0.0  ░░░░░░░░░░░░░░░░░░░░
  Örnek Tarihi                             %  0.0  ░░░░░░░░░░░░░░░░░░░░
  Yaş                                      %  0.0  ░░░░░░░░░░░░░░░░░░░░
  Cinsiyet                                 %  0.0  ░░░░░░░░░░░░░░░░░░░░
  Açlık Kan Şekeri (AKŞ)                   %  5.0  ░░░░░░░░░░░░░░░░░░░░
  Hemoglobin (Hb A1c)                      % 42.0  ████████░░░░░░░░░░░░
  Kolesterol (Total)                       % 48.0  █████████░░░░░░░░░░░
  Kolestrol (HDL)                          % 48.0  █████████░░░░░░░░░░░
  Kolestrol (LDL)                          % 48.6  █████████░░░░░░░░░░░
  Tokluk Kan Şekeri (TKŞ) 1.Saat           % 97.5  ███████████████████░
  Tokluk Kan Şekeri (TKŞ) 2.Saat           % 97.3  █████████

In [2]:
# ============================================================
# HÜCRE 2: Sayısal Temizlik ve Hasta Bazında Birleştirme
# ============================================================

def clean_numeric(val):
    """Virgüllü sayılar, metin karışıklıkları ve boşlukları temizler."""
    if pd.isna(val): return np.nan
    if isinstance(val, (int, float)): return float(val)
    val = str(val).replace(',', '.').strip()
    val = ''.join(c for c in val if c.isdigit() or c == '.')
    try: return float(val)
    except: return np.nan

def clean_age(val):
    """'79 yıl' → 79 gibi yaş temizliği."""
    if pd.isna(val): return np.nan
    if isinstance(val, (int, float)): return float(val)
    val = str(val).replace(' yıl', '').replace(',', '.').strip()
    val = ''.join(c for c in val if c.isdigit() or c == '.')
    try: return float(val)
    except: return np.nan

df = df_raw.copy()

# --- Tip Dönüşümleri ---
df['Yaş']      = df['Yaş'].apply(clean_age)
df['Cinsiyet'] = df['Cinsiyet'].map({'Erkek': 1, 'Kadın': 0})

numeric_cols = [
    'Açlık Kan Şekeri (AKŞ)',
    'Hemoglobin (Hb A1c)',
    'Tokluk Kan Şekeri (TKŞ) 1.Saat',
    'Tokluk Kan Şekeri (TKŞ) 2.Saat',
]
for col in numeric_cols:
    df[col] = df[col].apply(clean_numeric)

# --- TKŞ: 1. ve 2. saatin max'ını al (hangisi varsa) ---
df['TKS'] = df[['Tokluk Kan Şekeri (TKŞ) 1.Saat',
                 'Tokluk Kan Şekeri (TKŞ) 2.Saat']].max(axis=1)

# --- Hasta Bazında Birleştirme ---
# Her hasta için: HbA1c → ortalama (birden fazla ölçüm varsa)
#                 AKŞ   → ortalama
#                 TKŞ   → mevcut ölçümlerin ortalaması
#                 Yaş, Cinsiyet → ilk kayıt (değişmez demografik)
df_patient = df.groupby('Hasta No').agg(
    Yas      = ('Yaş',                         'first'),
    Cinsiyet = ('Cinsiyet',                     'first'),
    AKS_mean = ('Açlık Kan Şekeri (AKŞ)',       'mean'),
    AKS_max  = ('Açlık Kan Şekeri (AKŞ)',       'max'),   # zirve AKŞ de anlamlı
    TKS_mean = ('TKS',                          'mean'),
    HbA1c    = ('Hemoglobin (Hb A1c)',          'mean'),
    n_kayit  = ('Hasta No',                     'count'), # kaç ölçüm var
).reset_index()

# Sadece HbA1c'si olan hastaları al
df_patient = df_patient.dropna(subset=['HbA1c'])

print("=" * 55)
print("  HASTA BAZINDA BİRLEŞTİRME RAPORU")
print("=" * 55)
print(f"  Toplam hasta (HbA1c'li) : {len(df_patient):,}")
print(f"  Birden fazla kaydı olan : {(df_patient['n_kayit'] > 1).sum():,}")
print("=" * 55)

print("\n📋 Eksik Veri (Hasta Bazında):")
print("-" * 40)
for col in ['Yas', 'Cinsiyet', 'AKS_mean', 'AKS_max', 'TKS_mean', 'HbA1c']:
    miss = df_patient[col].isna().sum()
    pct  = miss / len(df_patient) * 100
    print(f"  {col:<15}: {miss:>5} eksik  (%{pct:.1f})")

print("\n📊 İstatistiksel Özet:")
print(df_patient[['Yas', 'AKS_mean', 'AKS_max', 'TKS_mean', 'HbA1c']].describe().round(2).to_string())

print("\n✅ Hücre 2 tamamlandı. Çıktıyı paylaşın → Hücre 3'e geçelim.")

  HASTA BAZINDA BİRLEŞTİRME RAPORU
  Toplam hasta (HbA1c'li) : 16,953
  Birden fazla kaydı olan : 8,170

📋 Eksik Veri (Hasta Bazında):
----------------------------------------
  Yas            :     0 eksik  (%0.0)
  Cinsiyet       :     0 eksik  (%0.0)
  AKS_mean       :   573 eksik  (%3.4)
  AKS_max        :   573 eksik  (%3.4)
  TKS_mean       : 16007 eksik  (%94.4)
  HbA1c          :     0 eksik  (%0.0)

📊 İstatistiksel Özet:
            Yas  AKS_mean   AKS_max  TKS_mean     HbA1c
count  16953.00  16380.00  16380.00    946.00  16953.00
mean      49.58    110.77    118.33    137.56      6.15
std       29.79     47.77     59.72     62.13      1.59
min       15.00     27.30     27.30     56.40      0.00
25%       37.00     85.90     87.80     98.36      5.27
50%       50.00     94.10     97.10    119.40      5.63
75%       61.00    113.30    120.30    153.89      6.40
max     1710.00    662.70   1095.20    652.60     17.80

✅ Hücre 2 tamamlandı. Çıktıyı paylaşın → Hücre 3'e geçelim.


In [3]:
# ============================================================
# HÜCRE 3: Outlier Temizliği ve Medikal Sınır Filtreleme
# ============================================================

df_clean = df_patient.copy()

print("=" * 55)
print("  OUTLIER TEMİZLİĞİ — ÖNCE / SONRA")
print("=" * 55)

# --- Medikal Referans Sınırları ---
# Kaynak: Klinik biyokimya standartları
LIMITS = {
    'Yas'     : (15, 110),    # 15 altı pediatrik → kapsam dışı, 110 üstü imkânsız
    'AKS_mean': (40, 600),    # mg/dL — 40 altı ölümcül hipoglisemi, 600 üstü laboratuvar hatası
    'AKS_max' : (40, 600),
    'TKS_mean': (40, 600),
    'HbA1c'   : (3.5, 16.0), # % — 3.5 altı ölümcül, 16 üstü laboratuvar hatası
}

before = len(df_clean)

for col, (lo, hi) in LIMITS.items():
    if col not in df_clean.columns:
        continue
    mask_out = df_clean[col].notna() & ((df_clean[col] < lo) | (df_clean[col] > hi))
    n_out = mask_out.sum()
    if n_out > 0:
        print(f"  {col:<15}: {n_out:>4} outlier tespit edildi "
              f"(sınır: {lo}–{hi})")
    # Outlier satırı tamamen çıkar (hedef değişken veya temel feature hatalıysa)
    if col in ['HbA1c', 'Yas']:
        df_clean = df_clean[~mask_out]
    else:
        # Feature outlier → NaN yap, sonraki adımda impute edilecek
        df_clean.loc[mask_out, col] = np.nan

after = len(df_clean)
print(f"\n  Çıkarılan hasta : {before - after:>4}")
print(f"  Kalan hasta     : {after:>4}")
print("=" * 55)

# --- TKŞ için "varlık" flag'i ---
# TKŞ %94 eksik → sayısal değeri kullan ama "ölçüm yapılmış mı" bilgisini de tut
df_clean['TKS_var'] = df_clean['TKS_mean'].notna().astype(int)

print("\n📊 Temizlik Sonrası İstatistikler:")
print(df_clean[['Yas', 'AKS_mean', 'AKS_max', 'TKS_mean', 'HbA1c']].describe().round(2).to_string())

print("\n📊 TKŞ Ölçümü Olan / Olmayan Dağılımı:")
tkss = df_clean['TKS_var'].value_counts()
print(f"  TKŞ Ölçümü YOK : {tkss.get(0, 0):,} hasta")
print(f"  TKŞ Ölçümü VAR : {tkss.get(1, 0):,} hasta")

print("\n📊 HbA1c Dağılımı (Klinik Kategoriler):")
bins   = [0, 5.7, 6.5, 16]
labels = ['Normal (<5.7)', 'Pre-Diyabet (5.7–6.5)', 'Diyabet (≥6.5)']
df_clean['HbA1c_kategori'] = pd.cut(df_clean['HbA1c'], bins=bins, labels=labels)
dist = df_clean['HbA1c_kategori'].value_counts()
for lbl in labels:
    n   = dist.get(lbl, 0)
    pct = n / len(df_clean) * 100
    print(f"  {lbl:<28}: {n:>5} hasta  (%{pct:.1f})")

print("\n✅ Hücre 3 tamamlandı. Çıktıyı paylaşın → Hücre 4'e geçelim.")

  OUTLIER TEMİZLİĞİ — ÖNCE / SONRA
  Yas            :   10 outlier tespit edildi (sınır: 15–110)
  AKS_mean       :    6 outlier tespit edildi (sınır: 40–600)
  AKS_max        :   11 outlier tespit edildi (sınır: 40–600)
  TKS_mean       :    2 outlier tespit edildi (sınır: 40–600)
  HbA1c          :   11 outlier tespit edildi (sınır: 3.5–16.0)

  Çıkarılan hasta :   21
  Kalan hasta     : 16932

📊 Temizlik Sonrası İstatistikler:
            Yas  AKS_mean   AKS_max  TKS_mean     HbA1c
count  16932.00  16354.00  16350.00    943.00  16932.00
mean      49.16    110.65    118.02    136.58      6.15
std       15.86     47.16     58.09     57.91      1.58
min       15.00     41.20     41.20     56.40      3.60
25%       37.00     85.90     87.80     98.38      5.27
50%       50.00     94.10     97.10    119.10      5.64
75%       61.00    113.22    120.20    153.57      6.40
max       92.00    599.30    599.30    432.10     15.90

📊 TKŞ Ölçümü Olan / Olmayan Dağılımı:
  TKŞ Ölçümü YOK : 15,9

In [4]:
# ============================================================
# HÜCRE 4: Feature Mühendisliği ve Veri Bölme
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

df_feat = df_clean.copy()

# -------------------------------------------------------
# A) AKŞ Eksiklerini Medyan ile Doldur (sadece %3.4 eksik)
# -------------------------------------------------------
aks_median = df_feat['AKS_mean'].median()
df_feat['AKS_mean'] = df_feat['AKS_mean'].fillna(aks_median)
df_feat['AKS_max']  = df_feat['AKS_max'].fillna(df_feat['AKS_mean'])

# -------------------------------------------------------
# B) Klinik Anlamlı Feature Mühendisliği
# -------------------------------------------------------

# 1. AKŞ Kategorik Riski (WHO sınırları: <100 normal, 100-125 pre, ≥126 diyabetik AKŞ)
df_feat['AKS_risk'] = pd.cut(
    df_feat['AKS_mean'],
    bins=[0, 100, 125, 600],
    labels=[0, 1, 2]
).astype(float)

# 2. AKŞ Volatilitesi: max - mean farkı (birden fazla ölçümü olan hastalarda anlamlı)
df_feat['AKS_volatilite'] = df_feat['AKS_max'] - df_feat['AKS_mean']

# 3. Yaş Grubu (endokrin açıdan anlamlı eşikler)
df_feat['Yas_grup'] = pd.cut(
    df_feat['Yas'],
    bins=[0, 30, 45, 60, 75, 110],
    labels=[0, 1, 2, 3, 4]
).astype(float)

# 4. AKŞ × Yaş etkileşimi (yaşlılarda aynı AKŞ daha yüksek HbA1c riski taşır)
df_feat['AKS_x_Yas'] = df_feat['AKS_mean'] * df_feat['Yas'] / 1000  # normalize

# 5. TKŞ: sadece ölçüm olan hastalarda gerçek değeri, yoksa NaN — XGBoost NaN'ı işler
#    Ayrıca "TKS_var" flag'i zaten var (Hücre 3'te eklendi)
df_feat['TKS_input'] = df_feat['TKS_mean']  # NaN olan yerler olduğu gibi kalır

# 6. Birden fazla ölçümü olan hasta (kronik takip → daha güvenilir)
df_feat['coklu_kayit'] = (df_clean.loc[df_feat.index, 'n_kayit'] > 1).astype(int) \
    if 'n_kayit' in df_clean.columns else 0

# -------------------------------------------------------
# C) Final Feature Seti
# -------------------------------------------------------
FEATURES = [
    'Yas',            # Temel demografik
    'Cinsiyet',       # Temel demografik
    'AKS_mean',       # Ana klinik gösterge
    'AKS_max',        # Zirve AKŞ (en kötü ölçüm)
    'AKS_risk',       # WHO kategorik riski
    'AKS_volatilite', # Ölçümler arası fark
    'Yas_grup',       # Yaş grubu
    'AKS_x_Yas',      # Etkileşim terimi
    'TKS_input',      # TKŞ (varsa gerçek, yoksa NaN → XGBoost halleder)
    'TKS_var',        # TKŞ ölçümü yapılmış mı?
    'coklu_kayit',    # Birden fazla kayıt var mı?
]
TARGET = 'HbA1c'

X = df_feat[FEATURES]
y = df_feat[TARGET]

# -------------------------------------------------------
# D) Train / Test Bölme — Stratified (HbA1c kategorisine göre)
# -------------------------------------------------------
strat_labels = df_feat['HbA1c_kategori'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=strat_labels
)

print("=" * 55)
print("  FEATURE MÜHENDİSLİĞİ ve VERİ BÖLME RAPORU")
print("=" * 55)
print(f"  Toplam Feature Sayısı : {len(FEATURES)}")
print(f"  Eğitim Seti           : {len(X_train):,} hasta")
print(f"  Test Seti             : {len(X_test):,} hasta")
print("=" * 55)

print("\n📋 Feature Listesi:")
for i, f in enumerate(FEATURES, 1):
    nan_train = X_train[f].isna().sum()
    print(f"  {i:>2}. {f:<20} — eğitimde {nan_train} NaN")

print("\n📊 Hedef Değişken (HbA1c) Dağılımı:")
print(f"  Eğitim → Mean: {y_train.mean():.3f}  Std: {y_train.std():.3f}")
print(f"  Test   → Mean: {y_test.mean():.3f}  Std: {y_test.std():.3f}")

print("\n✅ Hücre 4 tamamlandı. Çıktıyı paylaşın → Hücre 5'e geçelim.")

  FEATURE MÜHENDİSLİĞİ ve VERİ BÖLME RAPORU
  Toplam Feature Sayısı : 11
  Eğitim Seti           : 13,545 hasta
  Test Seti             : 3,387 hasta

📋 Feature Listesi:
   1. Yas                  — eğitimde 0 NaN
   2. Cinsiyet             — eğitimde 0 NaN
   3. AKS_mean             — eğitimde 0 NaN
   4. AKS_max              — eğitimde 0 NaN
   5. AKS_risk             — eğitimde 0 NaN
   6. AKS_volatilite       — eğitimde 0 NaN
   7. Yas_grup             — eğitimde 0 NaN
   8. AKS_x_Yas            — eğitimde 0 NaN
   9. TKS_input            — eğitimde 12796 NaN
  10. TKS_var              — eğitimde 0 NaN
  11. coklu_kayit          — eğitimde 0 NaN

📊 Hedef Değişken (HbA1c) Dağılımı:
  Eğitim → Mean: 6.154  Std: 1.572
  Test   → Mean: 6.145  Std: 1.588

✅ Hücre 4 tamamlandı. Çıktıyı paylaşın → Hücre 5'e geçelim.


In [12]:
# ============================================================
# HÜCRE 5: XGBoost + Optuna Hyperparameter Optimizasyonu
# ============================================================
# İlk çalıştırmada eksik kütüphane varsa:
%pip install optuna xgboost scikit-learn
# ============================================================
import optuna
import xgboost as xgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold
import time

optuna.logging.set_verbosity(optuna.logging.WARNING)

# -------------------------------------------------------
# A) Optuna Objective — 5-Fold CV üzerinden R² maksimize et
# -------------------------------------------------------
def objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 400, 1200),
        'max_depth'         : trial.suggest_int('max_depth', 3, 7),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight'  : trial.suggest_int('min_child_weight', 3, 20),
        'gamma'             : trial.suggest_float('gamma', 0.0, 2.0),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 0.5, 5.0),
        'objective'         : 'reg:squarederror',
        'tree_method'       : 'hist',
        'random_state'      : 42,
        'n_jobs'            : -1,
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    r2_scores = []

    for train_idx, val_idx in kf.split(X_train):
        Xtr, Xval = X_train.iloc[train_idx], X_train.iloc[val_idx]
        ytr, yval = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(
            Xtr, ytr,
            eval_set=[(Xval, yval)],
            verbose=False,
        )
        preds = model.predict(Xval)
        r2_scores.append(r2_score(yval, preds))

    return np.mean(r2_scores)

# -------------------------------------------------------
# B) Optimizasyon — 60 deneme
# -------------------------------------------------------
print("⏳ Optuna optimizasyonu başlıyor (60 deneme, ~2-4 dk)...")
t0 = time.time()

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=60, show_progress_bar=False)

elapsed = time.time() - t0
print(f"✅ Optimizasyon tamamlandı ({elapsed:.0f} sn)")
print(f"\n🏆 En İyi CV R²  : %{study.best_value * 100:.4f}")
print(f"\n🔧 En İyi Parametreler:")
for k, v in study.best_params.items():
    print(f"   {k:<22}: {v}")

# -------------------------------------------------------
# C) Final Model — En iyi parametrelerle tüm eğitim seti
# -------------------------------------------------------
best_params = study.best_params.copy()
best_params.update({
    'objective'  : 'reg:squarederror',
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs'     : -1,
})

print("\n⏳ Final model eğitiliyor (tüm eğitim seti)...")
final_model = xgb.XGBRegressor(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

# -------------------------------------------------------
# D) Test Seti Değerlendirmesi
# -------------------------------------------------------
y_pred_test  = final_model.predict(X_test)
y_pred_train = final_model.predict(X_train)

r2_test  = r2_score(y_test,  y_pred_test)
r2_train = r2_score(y_train, y_pred_train)
mae      = mean_absolute_error(y_test, y_pred_test)
rmse     = np.sqrt(mean_squared_error(y_test, y_pred_test))
gap      = abs(r2_train - r2_test)

print("\n" + "=" * 55)
print("  PERFORMANS RAPORU")
print("=" * 55)
print(f"  CV  R²  (Optuna)  : %{study.best_value * 100:.2f}")
print(f"  Eğitim R²         : %{r2_train * 100:.2f}")
print(f"  Test   R²         : %{r2_test  * 100:.2f}")
print(f"  Genelleme Farkı   : %{gap * 100:.2f}  {'✅ İyi' if gap < 0.03 else '⚠️ Dikkat'}")
print(f"  MAE               : {mae:.4f} HbA1c birimi")
print(f"  RMSE              : {rmse:.4f} HbA1c birimi")
print("=" * 55)

print("\n✅ Hücre 5 tamamlandı. Çıktıyı paylaşın → Hücre 6'ya geçelim.")

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
  Using cached optuna-4.8.0-py3-none-any.whl.metadata (17 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached mako-1.3.12-py3-none-any.whl.metadata (2.9 kB)
Using cached optuna-4.8.0-py3-none-any.whl (419 kB)
Using cached alembic-1.18.4-py3-none-any.whl (263 kB)
Using cached colorlog-6.10.1-py3-none-any.whl (11 kB)
Using cached mako-1.3.12-py3-none-any.whl (78 kB)

   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   ------------------------------ --------- 3/4 [optuna]


In [13]:
# ============================================================
# HÜCRE 6: Gelişmiş Feature Mühendisliği + Yeniden Eğitim
# ============================================================

df_feat2 = df_clean.copy()

# --- AKŞ Temizliği ---
aks_median = df_feat2['AKS_mean'].median()
df_feat2['AKS_mean'] = df_feat2['AKS_mean'].fillna(aks_median)
df_feat2['AKS_max']  = df_feat2['AKS_max'].fillna(df_feat2['AKS_mean'])

# ============================================================
# GELİŞMİŞ KLİNİK FEATURE'LAR
# ============================================================

# 1. AKŞ → tahmini HbA1c dönüşümü (Nathan 2008 formülü: eAG = 28.7×HbA1c − 46.7)
#    Tersine: HbA1c_est = (AKŞ + 46.7) / 28.7
df_feat2['HbA1c_est_AKS']  = (df_feat2['AKS_mean'] + 46.7) / 28.7
df_feat2['HbA1c_est_AKSmax'] = (df_feat2['AKS_max'] + 46.7) / 28.7

# 2. AKŞ log dönüşümü (sağa çarpık dağılımı normalize eder)
df_feat2['AKS_log']    = np.log1p(df_feat2['AKS_mean'])
df_feat2['AKS_kare']   = df_feat2['AKS_mean'] ** 2 / 10000  # scale

# 3. AKŞ WHO eşiği üzeri fazlalık (threshold feature)
df_feat2['AKS_fazla_100'] = np.maximum(0, df_feat2['AKS_mean'] - 100)
df_feat2['AKS_fazla_126'] = np.maximum(0, df_feat2['AKS_mean'] - 126)

# 4. Yaş × AKŞ etkileşimleri
df_feat2['AKS_x_Yas']     = df_feat2['AKS_mean'] * df_feat2['Yas'] / 1000
df_feat2['AKSmax_x_Yas']  = df_feat2['AKS_max']  * df_feat2['Yas'] / 1000

# 5. Yaş kategorik
df_feat2['Yas_grup'] = pd.cut(
    df_feat2['Yas'],
    bins=[0, 30, 45, 60, 75, 110],
    labels=[0, 1, 2, 3, 4]
).astype(float)

# 6. AKŞ risk kategorisi
df_feat2['AKS_risk'] = pd.cut(
    df_feat2['AKS_mean'],
    bins=[0, 100, 125, 600],
    labels=[0, 1, 2]
).astype(float)

# 7. Volatilite (çoklu kayıt olan hastalarda max-mean farkı)
df_feat2['AKS_volatilite'] = df_feat2['AKS_max'] - df_feat2['AKS_mean']
df_feat2['AKS_vol_log']    = np.log1p(df_feat2['AKS_volatilite'])

# 8. Cinsiyet × Yaş (kadınlarda menopoz sonrası risk farklı)
df_feat2['Cinsiyet_x_Yas'] = df_feat2['Cinsiyet'] * df_feat2['Yas']

# 9. TKŞ feature'ları (varsa)
df_feat2['TKS_input']      = df_feat2['TKS_mean']
df_feat2['TKS_var']        = df_feat2['TKS_mean'].notna().astype(int)
df_feat2['TKS_log']        = np.log1p(df_feat2['TKS_mean'].fillna(0)) * df_feat2['TKS_var']
df_feat2['HbA1c_est_TKS']  = ((df_feat2['TKS_mean'].fillna(0) + 46.7) / 28.7) * df_feat2['TKS_var']

# 10. Çoklu kayıt flag
df_feat2['coklu_kayit'] = (df_clean['n_kayit'] > 1).astype(int)
df_feat2['n_kayit_log'] = np.log1p(df_clean['n_kayit'])

# ============================================================
FEATURES2 = [
    # Temel
    'Yas', 'Cinsiyet',
    # AKŞ — ham
    'AKS_mean', 'AKS_max',
    # AKŞ — dönüşümler
    'AKS_log', 'AKS_kare',
    'AKS_fazla_100', 'AKS_fazla_126',
    # Klinik tahmin
    'HbA1c_est_AKS', 'HbA1c_est_AKSmax',
    # Kategorik
    'AKS_risk', 'Yas_grup',
    # Etkileşimler
    'AKS_x_Yas', 'AKSmax_x_Yas', 'Cinsiyet_x_Yas',
    # Volatilite
    'AKS_volatilite', 'AKS_vol_log',
    # TKŞ
    'TKS_input', 'TKS_var', 'TKS_log', 'HbA1c_est_TKS',
    # Kayıt bilgisi
    'coklu_kayit', 'n_kayit_log',
]

X2 = df_feat2[FEATURES2]
y2 = df_feat2['HbA1c']

strat2 = df_feat2['HbA1c_kategori'].astype(str)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42, stratify=strat2
)

# ============================================================
# YENİDEN OPTİMİZASYON — Yeni feature setiyle
# ============================================================
print(f"✅ Yeni feature sayısı: {len(FEATURES2)}")
print(f"   Eğitim: {len(X2_train):,} | Test: {len(X2_test):,}")
print("\n⏳ Optuna optimizasyonu (80 deneme)...")

def objective2(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 500, 2000),
        'max_depth'        : trial.suggest_int('max_depth', 3, 8),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight' : trial.suggest_int('min_child_weight', 3, 25),
        'gamma'            : trial.suggest_float('gamma', 0.0, 3.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 0.0, 3.0),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0.5, 6.0),
        'max_delta_step'   : trial.suggest_int('max_delta_step', 0, 5),
        'objective'        : 'reg:squarederror',
        'tree_method'      : 'hist',
        'random_state'     : 42,
        'n_jobs'           : -1,
    }
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr, vl in kf.split(X2_train):
        Xtr, Xvl = X2_train.iloc[tr], X2_train.iloc[vl]
        ytr, yvl = y2_train.iloc[tr], y2_train.iloc[vl]
        m = xgb.XGBRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xvl, yvl)], verbose=False)
        scores.append(r2_score(yvl, m.predict(Xvl)))
    return np.mean(scores)

t0 = time.time()
study2 = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=42))
study2.optimize(objective2, n_trials=80, show_progress_bar=False)
print(f"✅ Tamamlandı ({time.time()-t0:.0f} sn)")

# Final model
bp2 = study2.best_params.copy()
bp2.update({'objective':'reg:squarederror','tree_method':'hist',
            'random_state':42,'n_jobs':-1})

final_model2 = xgb.XGBRegressor(**bp2)
final_model2.fit(X2_train, y2_train,
                 eval_set=[(X2_test, y2_test)], verbose=False)

yp_test  = final_model2.predict(X2_test)
yp_train = final_model2.predict(X2_train)

r2_te = r2_score(y2_test,  yp_test)
r2_tr = r2_score(y2_train, yp_train)
mae2  = mean_absolute_error(y2_test, yp_test)
rmse2 = np.sqrt(mean_squared_error(y2_test, yp_test))
gap2  = abs(r2_tr - r2_te)

print("\n" + "=" * 55)
print("  KARŞILAŞTIRMALI PERFORMANS RAPORU")
print("=" * 55)
print(f"  {'':30} Önceki    Şimdi")
print(f"  {'-'*50}")
print(f"  {'CV R²':<30} %67.89  → %{study2.best_value*100:.2f}")
print(f"  {'Eğitim R²':<30} %70.08  → %{r2_tr*100:.2f}")
print(f"  {'Test R²':<30} %66.48  → %{r2_te*100:.2f}")
print(f"  {'Genelleme Farkı':<30} %3.60   → %{gap2*100:.2f}  {'✅' if gap2<0.03 else '⚠️'}")
print(f"  {'MAE':<30} 0.5525  → {mae2:.4f}")
print(f"  {'RMSE':<30} 0.9193  → {rmse2:.4f}")
print("=" * 55)

print("\n✅ Hücre 6 tamamlandı. Çıktıyı paylaşın → Hücre 7'ye geçelim.")

✅ Yeni feature sayısı: 23
   Eğitim: 13,545 | Test: 3,387

⏳ Optuna optimizasyonu (80 deneme)...
✅ Tamamlandı (595 sn)

  KARŞILAŞTIRMALI PERFORMANS RAPORU
                                 Önceki    Şimdi
  --------------------------------------------------
  CV R²                          %67.89  → %67.88
  Eğitim R²                      %70.08  → %71.34
  Test R²                        %66.48  → %66.49
  Genelleme Farkı                %3.60   → %4.85  ⚠️
  MAE                            0.5525  → 0.5530
  RMSE                           0.9193  → 0.9192

✅ Hücre 6 tamamlandı. Çıktıyı paylaşın → Hücre 7'ye geçelim.


In [14]:
# ============================================================
# HÜCRE 7: Cascade Model — Önce Sınıflandır, Sonra Regresyon
# ============================================================
# Strateji:
#   Adım 1 → XGBoost Classifier: Normal / Pre-Diyabet / Diyabet tahmin et
#   Adım 2 → Her grup için ayrı XGBoost Regressor
#   Sonuç  → Tahmin edilen grubun regressor'ından HbA1c al
# ============================================================
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold, train_test_split
import xgboost as xgb
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# -------------------------------------------------------
# Veriyi yeniden hazırla (Hücre 3'ten df_clean kullan)
# -------------------------------------------------------
df_c = df_clean.copy()
df_c['AKS_mean'] = df_c['AKS_mean'].fillna(df_c['AKS_mean'].median())
df_c['AKS_max']  = df_c['AKS_max'].fillna(df_c['AKS_mean'])

# Feature seti (sade — overfitting riskini düşür)
def make_features(d):
    out = d.copy()
    out['HbA1c_est']      = (out['AKS_mean'] + 46.7) / 28.7
    out['HbA1c_est_max']  = (out['AKS_max']  + 46.7) / 28.7
    out['AKS_log']        = np.log1p(out['AKS_mean'])
    out['AKS_fazla_100']  = np.maximum(0, out['AKS_mean'] - 100)
    out['AKS_fazla_126']  = np.maximum(0, out['AKS_mean'] - 126)
    out['AKS_x_Yas']      = out['AKS_mean'] * out['Yas'] / 1000
    out['AKS_volatilite'] = out['AKS_max'] - out['AKS_mean']
    out['Yas_grup']       = pd.cut(out['Yas'],bins=[0,30,45,60,75,110],labels=[0,1,2,3,4]).astype(float)
    out['AKS_risk']       = pd.cut(out['AKS_mean'],bins=[0,100,125,600],labels=[0,1,2]).astype(float)
    out['Cinsiyet_x_Yas'] = out['Cinsiyet'] * out['Yas']
    out['TKS_input']      = out['TKS_mean']
    out['TKS_var']        = out['TKS_mean'].notna().astype(int)
    out['coklu_kayit']    = (out['n_kayit'] > 1).astype(int)
    return out

df_c = make_features(df_c)

FEATS = ['Yas','Cinsiyet','AKS_mean','AKS_max','HbA1c_est','HbA1c_est_max',
         'AKS_log','AKS_fazla_100','AKS_fazla_126','AKS_x_Yas',
         'AKS_volatilite','Yas_grup','AKS_risk','Cinsiyet_x_Yas',
         'TKS_input','TKS_var','coklu_kayit']

# Etiketler
df_c['label_cat'] = pd.cut(df_c['HbA1c'],bins=[0,5.7,6.5,16],labels=[0,1,2]).astype(int)

X_all = df_c[FEATS]
y_reg = df_c['HbA1c']
y_clf = df_c['label_cat']

# Stratified split
X_tr, X_te, y_tr_reg, y_te_reg, y_tr_clf, y_te_clf = train_test_split(
    X_all, y_reg, y_clf,
    test_size=0.20, random_state=42,
    stratify=y_clf
)

print(f"Eğitim: {len(X_tr):,}  |  Test: {len(X_te):,}")

# -------------------------------------------------------
# ADIM 1: Sınıflandırıcı — grubu tahmin et
# -------------------------------------------------------
clf = xgb.XGBClassifier(
    n_estimators=600, max_depth=4, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.7,
    min_child_weight=5, gamma=0.5,
    reg_alpha=0.3, reg_lambda=2.0,
    objective='multi:softmax', num_class=3,
    tree_method='hist', random_state=42, n_jobs=-1,
    eval_metric='mlogloss'
)
clf.fit(X_tr, y_tr_clf, eval_set=[(X_te, y_te_clf)], verbose=False)

clf_pred_tr = clf.predict(X_tr)
clf_pred_te = clf.predict(X_te)
clf_acc = (clf_pred_te == y_te_clf).mean()
print(f"\n📊 Sınıflandırıcı Test Doğruluğu: %{clf_acc*100:.2f}")

# -------------------------------------------------------
# ADIM 2: Her grup için ayrı Regressor
# -------------------------------------------------------
regressors = {}
group_names = {0: 'Normal', 1: 'Pre-Diyabet', 2: 'Diyabet'}

print("\n📊 Grup Bazlı Regressor Eğitimi:")
print("-" * 45)

for g in [0, 1, 2]:
    mask_tr = (y_tr_clf == g)
    mask_te = (y_te_clf == g)

    Xg_tr = X_tr[mask_tr]
    yg_tr = y_tr_reg[mask_tr]

    reg = xgb.XGBRegressor(
        n_estimators=800, max_depth=4, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.7,
        min_child_weight=5, gamma=0.5,
        reg_alpha=0.3, reg_lambda=2.5,
        objective='reg:squarederror',
        tree_method='hist', random_state=42, n_jobs=-1,
    )
    reg.fit(Xg_tr, yg_tr, verbose=False)
    regressors[g] = reg

    # Gerçek grup test R²
    Xg_te = X_te[mask_te]
    yg_te = y_te_reg[mask_te]
    r2g   = r2_score(yg_te, reg.predict(Xg_te))
    print(f"  {group_names[g]:<15}: n_train={mask_tr.sum():>5,}  Test R²=%{r2g*100:.2f}")

# -------------------------------------------------------
# ADIM 3: Cascade tahmin — sınıflandırıcının grubunu kullan
# -------------------------------------------------------
def cascade_predict(X, clf, regressors):
    groups = clf.predict(X)
    preds  = np.zeros(len(X))
    for g, reg in regressors.items():
        mask = (groups == g)
        if mask.sum() > 0:
            preds[mask] = reg.predict(X[mask])
    return preds, groups

y_pred_cascade_te, grp_te = cascade_predict(X_te, clf, regressors)
y_pred_cascade_tr, _      = cascade_predict(X_tr, clf, regressors)

r2_cascade_te = r2_score(y_te_reg, y_pred_cascade_te)
r2_cascade_tr = r2_score(y_tr_reg, y_pred_cascade_tr)
mae_c  = mean_absolute_error(y_te_reg, y_pred_cascade_te)
rmse_c = np.sqrt(mean_squared_error(y_te_reg, y_pred_cascade_te))
gap_c  = abs(r2_cascade_tr - r2_cascade_te)

# Baseline (Hücre 5 sonucu)
print("\n" + "=" * 58)
print("  KARŞILAŞTIRMALI SONUÇ")
print("=" * 58)
print(f"  {'Metrik':<25} {'Baseline':>10}  {'Cascade':>10}")
print(f"  {'-'*50}")
print(f"  {'Test R²':<25} {'%66.48':>10}  %{r2_cascade_te*100:>6.2f}")
print(f"  {'Eğitim R²':<25} {'%70.08':>10}  %{r2_cascade_tr*100:>6.2f}")
print(f"  {'Genelleme Farkı':<25} {'%3.60':>10}  %{gap_c*100:>6.2f}  {'✅' if gap_c<0.03 else '⚠️'}")
print(f"  {'MAE':<25} {'0.5525':>10}  {mae_c:>10.4f}")
print(f"  {'RMSE':<25} {'0.9193':>10}  {rmse_c:>10.4f}")
print("=" * 58)

print("\n✅ Hücre 7 tamamlandı. Çıktıyı paylaşın → Hücre 8'e geçelim.")

Eğitim: 13,545  |  Test: 3,387

📊 Sınıflandırıcı Test Doğruluğu: %76.17

📊 Grup Bazlı Regressor Eğitimi:
---------------------------------------------
  Normal         : n_train=7,502  Test R²=%11.80
  Pre-Diyabet    : n_train=2,942  Test R²=%16.93
  Diyabet        : n_train=3,101  Test R²=%34.67

  KARŞILAŞTIRMALI SONUÇ
  Metrik                      Baseline     Cascade
  --------------------------------------------------
  Test R²                       %66.48  % 63.82
  Eğitim R²                     %70.08  % 69.54
  Genelleme Farkı                %3.60  %  5.72  ⚠️
  MAE                           0.5525      0.6079
  RMSE                          0.9193      0.9531

✅ Hücre 7 tamamlandı. Çıktıyı paylaşın → Hücre 8'e geçelim.


In [15]:
# ============================================================
# HÜCRE 8: TKŞ Tahmincisi → HbA1c Modeli
# ============================================================
# Strateji:
#   A) 943 gerçek TKŞ'li hastayla TKŞ tahmincisi eğit
#   B) Tüm hastalara TKŞ tahmini yap (estimated TKŞ)
#   C) Gerçek TKŞ varsa onu, yoksa tahminiyi kullan
#   D) Zenginleştirilmiş feature setiyle ana modeli eğit
# ============================================================
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# --- Veri hazırlık (df_clean'den devam) ---
df_m = df_clean.copy()
df_m['AKS_mean'] = df_m['AKS_mean'].fillna(df_m['AKS_mean'].median())
df_m['AKS_max']  = df_m['AKS_max'].fillna(df_m['AKS_mean'])

# ============================================================
# A) TKŞ TAHMİNCİSİ — sadece gerçek TKŞ'li 943 hasta
# ============================================================
tks_train_df = df_m[df_m['TKS_mean'].notna()].copy()

X_tks = tks_train_df[['AKS_mean', 'AKS_max', 'Yas', 'Cinsiyet']]
y_tks = tks_train_df['TKS_mean']

# 5-fold CV ile TKŞ tahmincisinin gücünü ölç
tks_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.8,
    min_child_weight=5, reg_alpha=0.3, reg_lambda=2.0,
    objective='reg:squarederror', tree_method='hist',
    random_state=42, n_jobs=-1,
)

cv_r2_tks = cross_val_score(tks_model, X_tks, y_tks, cv=5, scoring='r2')
print(f"📊 TKŞ Tahmincisi CV R²: {cv_r2_tks.mean():.4f} ± {cv_r2_tks.std():.4f}")

# Tüm TKŞ verisiyle final TKŞ modelini eğit
tks_model.fit(X_tks, y_tks, verbose=False)

# ============================================================
# B) TÜM HASTALARA TKŞ TAHMİNİ YAP
# ============================================================
X_all_tks = df_m[['AKS_mean', 'AKS_max', 'Yas', 'Cinsiyet']]
tks_estimated = tks_model.predict(X_all_tks)

# Gerçek varsa gerçeği, yoksa tahmini kullan
df_m['TKS_final']    = np.where(
    df_m['TKS_mean'].notna(),
    df_m['TKS_mean'],          # gerçek TKŞ
    tks_estimated              # tahmini TKŞ
)
df_m['TKS_gercek']   = df_m['TKS_mean'].notna().astype(int)  # kaynak flag

print(f"   Gerçek TKŞ kullanan  : {df_m['TKS_gercek'].sum():,} hasta")
print(f"   Tahmini TKŞ kullanan : {(~df_m['TKS_mean'].notna()).sum():,} hasta")

# ============================================================
# C) ZENGİNLEŞTİRİLMİŞ FEATURE SETİ
# ============================================================
# Nathan formülü: eAG = 28.7×HbA1c − 46.7 → HbA1c = (eAG+46.7)/28.7
df_m['HbA1c_est_AKS']  = (df_m['AKS_mean']    + 46.7) / 28.7
df_m['HbA1c_est_AKSmax']= (df_m['AKS_max']    + 46.7) / 28.7
df_m['HbA1c_est_TKS']  = (df_m['TKS_final']   + 46.7) / 28.7  # ← yeni güçlü feature

# TKŞ/AKŞ oranı — postprandial amplifikasyon
df_m['TKS_AKS_oran']   = df_m['TKS_final'] / (df_m['AKS_mean'] + 1e-5)

# Ağırlıklı glukoz skoru (AKŞ %40 + TKŞ %60 — OGTT mantığı)
df_m['Glukoz_agirlikli'] = 0.4 * df_m['AKS_mean'] + 0.6 * df_m['TKS_final']
df_m['HbA1c_est_agirlikli'] = (df_m['Glukoz_agirlikli'] + 46.7) / 28.7

df_m['AKS_log']         = np.log1p(df_m['AKS_mean'])
df_m['TKS_log']         = np.log1p(df_m['TKS_final'])
df_m['AKS_fazla_100']   = np.maximum(0, df_m['AKS_mean'] - 100)
df_m['AKS_fazla_126']   = np.maximum(0, df_m['AKS_mean'] - 126)
df_m['TKS_fazla_140']   = np.maximum(0, df_m['TKS_final'] - 140)  # OGTT eşiği
df_m['TKS_fazla_200']   = np.maximum(0, df_m['TKS_final'] - 200)  # diyabetik TKŞ
df_m['AKS_x_Yas']       = df_m['AKS_mean'] * df_m['Yas'] / 1000
df_m['TKS_x_Yas']       = df_m['TKS_final'] * df_m['Yas'] / 1000
df_m['AKS_volatilite']  = df_m['AKS_max'] - df_m['AKS_mean']
df_m['Yas_grup']        = pd.cut(df_m['Yas'],bins=[0,30,45,60,75,110],labels=[0,1,2,3,4]).astype(float)
df_m['AKS_risk']        = pd.cut(df_m['AKS_mean'],bins=[0,100,125,600],labels=[0,1,2]).astype(float)
df_m['coklu_kayit']     = (df_m['n_kayit'] > 1).astype(int)

FEATURES_FINAL = [
    'Yas', 'Cinsiyet',
    # AKŞ
    'AKS_mean', 'AKS_max', 'AKS_log',
    'AKS_fazla_100', 'AKS_fazla_126',
    'AKS_volatilite', 'AKS_risk',
    # TKŞ (gerçek + tahmini)
    'TKS_final', 'TKS_log',
    'TKS_fazla_140', 'TKS_fazla_200',
    'TKS_gercek',         # kaynak güvenilirlik flag
    # Klinik tahminler (Nathan formülü)
    'HbA1c_est_AKS', 'HbA1c_est_AKSmax',
    'HbA1c_est_TKS', 'HbA1c_est_agirlikli',
    # Oranlar ve etkileşimler
    'TKS_AKS_oran',
    'AKS_x_Yas', 'TKS_x_Yas',
    'Yas_grup',
    'coklu_kayit',
]

X_final = df_m[FEATURES_FINAL]
y_final = df_m['HbA1c']

strat_lbl = df_m['HbA1c_kategori'].astype(str)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_final, y_final, test_size=0.20,
    random_state=42, stratify=strat_lbl
)

# ============================================================
# D) ANA MODEL — Optuna ile optimize et
# ============================================================
import optuna, time
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_final(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 500, 2000),
        'max_depth'       : trial.suggest_int('max_depth', 3, 7),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 20),
        'gamma'           : trial.suggest_float('gamma', 0.0, 2.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 0.0, 3.0),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 0.5, 5.0),
        'objective': 'reg:squarederror', 'tree_method': 'hist',
        'random_state': 42, 'n_jobs': -1,
    }
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_i, vl_i in kf.split(X_tr):
        m = xgb.XGBRegressor(**params)
        m.fit(X_tr.iloc[tr_i], y_tr.iloc[tr_i],
              eval_set=[(X_tr.iloc[vl_i], y_tr.iloc[vl_i])], verbose=False)
        scores.append(r2_score(y_tr.iloc[vl_i], m.predict(X_tr.iloc[vl_i])))
    return np.mean(scores)

print("\n⏳ Optuna optimizasyonu (80 deneme)...")
t0 = time.time()
study_f = optuna.create_study(direction='maximize',
                               sampler=optuna.samplers.TPESampler(seed=42))
study_f.optimize(objective_final, n_trials=80, show_progress_bar=False)
print(f"✅ Tamamlandı ({time.time()-t0:.0f} sn)")

bp = study_f.best_params.copy()
bp.update({'objective':'reg:squarederror','tree_method':'hist','random_state':42,'n_jobs':-1})

final_m = xgb.XGBRegressor(**bp)
final_m.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

y_pred_te = final_m.predict(X_te)
y_pred_tr = final_m.predict(X_tr)

r2_te  = r2_score(y_te, y_pred_te)
r2_tr  = r2_score(y_tr, y_pred_tr)
mae_f  = mean_absolute_error(y_te, y_pred_te)
rmse_f = np.sqrt(mean_squared_error(y_te, y_pred_te))
gap_f  = abs(r2_tr - r2_te)

print("\n" + "=" * 58)
print("  KARŞILAŞTIRMALI SONUÇ")
print("=" * 58)
print(f"  {'Metrik':<25} {'Baseline':>10}  {'TKŞ+Model':>10}")
print(f"  {'-'*52}")
print(f"  {'CV R²':<25} {'%67.89':>10}  %{study_f.best_value*100:>6.2f}")
print(f"  {'Test R²':<25} {'%66.48':>10}  %{r2_te*100:>6.2f}")
print(f"  {'Eğitim R²':<25} {'%70.08':>10}  %{r2_tr*100:>6.2f}")
print(f"  {'Genelleme Farkı':<25} {'%3.60':>10}  %{gap_f*100:>6.2f}  {'✅' if gap_f<0.03 else '⚠️'}")
print(f"  {'MAE':<25} {'0.5525':>10}  {mae_f:>10.4f}")
print(f"  {'RMSE':<25} {'0.9193':>10}  {rmse_f:>10.4f}")
print("=" * 58)

print("\n✅ Hücre 8 tamamlandı. Çıktıyı paylaşın → Hücre 9'a geçelim.")

📊 TKŞ Tahmincisi CV R²: 0.5471 ± 0.0551
   Gerçek TKŞ kullanan  : 943 hasta
   Tahmini TKŞ kullanan : 15,989 hasta

⏳ Optuna optimizasyonu (80 deneme)...
✅ Tamamlandı (485 sn)

  KARŞILAŞTIRMALI SONUÇ
  Metrik                      Baseline   TKŞ+Model
  ----------------------------------------------------
  CV R²                         %67.89  % 67.80
  Test R²                       %66.48  % 66.40
  Eğitim R²                     %70.08  % 70.31
  Genelleme Farkı                %3.60  %  3.92  ⚠️
  MAE                           0.5525      0.5524
  RMSE                          0.9193      0.9205

✅ Hücre 8 tamamlandı. Çıktıyı paylaşın → Hücre 9'a geçelim.
